# Coverage control for the IVT comparison
| Figure | Description |
|--------|------------|
| **Supp. Fig. S11** | coverage-control figure (position, depth, imbalance, missed sites) |
| **Supp. Table S11** | false positives by transcript region |
| **Supp. Table S12** | false positives by depth and native–IVT imbalance |
| **Supp. Table S13** | site recovery by depth quartile |
| **Supp. Table S14** | the three sites no tool detected |

## 1. Inputs

| Input | Description |
|---|---|
| `inputs/depth_1000x/` | 12 `samtools depth` files — native and IVT, both rRNAs, 3 replicates, 1000× |
| `inputs/drummer_readcount/` | DRUMMER per-position read counts; consistency check only |
| `inputs/pinned/` | Calls, known sites, offsets and score-row decisions from the main benchmark |


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "inputs" / "pinned").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project directory (the one containing inputs/).")
INPUTS, OUTDIR = ROOT / "inputs", ROOT / "outputs"
OUTDIR.mkdir(exist_ok=True)

CORE = {"16s": (201, 1742), "23s": (201, 3104)}
REPS = ["rep1", "rep2", "rep3"]
SAMPLES = ["native", "ivt"]
THIRDS = ["5-prime", "middle", "3-prime"]
TLAB = {"5-prime": "5′ end", "middle": "middle", "3-prime": "3′ end"}
QUARTILES = ["Q1", "Q2", "Q3", "Q4"]
NEAR_TRUTH_NT = 5
MIN_OPPORTUNITIES = 500
SCOPE_EXACT = "exact-match benchmark universe"
SCOPE_PM5 = f"excluding ±{NEAR_TRUTH_NT} nt of a known site (sensitivity)"

NATIVE_C, IVT_C, GREY, INK = "#0072B2", "#D55E00", "#8A99A3", "#1F2D36"
HEAT = LinearSegmentedColormap.from_list("heat", ["#F4F8FB", "#9CC7E0", "#2E7EB0", "#0B3B57"])
pd.set_option("display.width", 180)
print(f"project directory: {ROOT.name}")
print(f"outputs: {OUTDIR.name}/")

In [ ]:
depth = pd.concat(
    [pd.read_csv(INPUTS / "depth_1000x" / f"{s}_{r}_{rep}.txt", sep="\t", header=None,
                 names=["contig", "pos", "depth"]).assign(sample=s, ref=r, rep=rep)
     for s in SAMPLES for r in CORE for rep in REPS], ignore_index=True)

per_file_contigs = depth.groupby(["sample", "ref", "rep"])["contig"].nunique()
if (per_file_contigs != 1).any():
    raise ValueError("A depth file contains more than one contig; check inputs/depth_1000x/.")

coverage = depth.pivot(index=["ref", "rep", "pos"], columns="sample",
                       values="depth").reset_index()
coverage.columns.name = None
coverage["limiting"] = coverage[SAMPLES].min(axis=1)
coverage["log2_ratio"] = np.log2((coverage["native"] + 1) / (coverage["ivt"] + 1))
coverage["abs_log2_ratio"] = coverage["log2_ratio"].abs()
if coverage[["native", "ivt"]].isna().any().any():
    raise ValueError("Native/IVT coverage pairing is incomplete.")
print(f"{len(coverage):,} position × replicate rows")
coverage.head()


## 2. DRUMMER depth check


In [ ]:
TAG = {"native": "MOD", "ivt": "UNMOD"}
checks = []
for s in SAMPLES:
    for r in CORE:
        for rep in REPS:
            rc = pd.read_csv(INPUTS / "drummer_readcount" / f"{r}_{rep}_{TAG[s]}.tsv",
                             sep="\t", header=None,
                             names=["contig", "pos", "base", "depth"]).set_index("pos")["depth"]
            mine = coverage.loc[(coverage["ref"] == r) & (coverage["rep"] == rep)] \
                           .set_index("pos")[s]
            both = pd.concat([rc.rename("tool"), mine.rename("file")], axis=1).dropna()
            rel = (both["tool"] - both["file"]) / both["tool"].replace(0, np.nan)
            checks.append({"sample": s, "ref": r, "rep": rep,
                           "mean_abs_rel_diff_pct": round(100 * rel.abs().mean(), 2)})

identity = pd.DataFrame(checks)
worst = identity["mean_abs_rel_diff_pct"].max()
if worst > 10:
    raise ValueError(f"Depth files differ from the tools' own read counts by {worst:.1f}%; "
                     "these are probably from a different subsampling run.")
print(f"largest mean difference: {worst:.2f}%  (limit 10%)")
identity.head()

## 3. Calls, known sites and offsets


In [ ]:
calls = pd.read_csv(INPUTS / "pinned" / "call_events.csv")
truth = pd.read_csv(INPUTS / "pinned" / "ground_truth_sites.csv")
tools = pd.read_csv(INPUTS / "pinned" / "tool_metadata.csv").sort_values("order")
offsets = pd.read_csv(INPUTS / "pinned" / "tool_reference_offsets.csv")
scores = pd.read_csv(
    INPUTS / "pinned" / "score_rows.csv.gz",
    usecols=["tool", "ref", "replicate", "canonical_pos", "scored_pos", "score_metric",
             "threshold", "called_positive", "threshold_roundtrip_rescued"],
).rename(columns={"replicate": "rep"})

keys = ["tool", "ref", "rep", "canonical_pos"]
if calls.duplicated(keys).any() or scores.duplicated(keys).any():
    raise ValueError("Pinned call/score keys are not unique.")
call_keys = set(map(tuple, calls[keys].to_numpy()))
score_positive_keys = set(map(tuple, scores.loc[scores["called_positive"], keys].to_numpy()))
if call_keys != score_positive_keys:
    raise ValueError("Pinned call events disagree with pinned score decisions.")
print(f"{len(calls):,} calls | {len(truth)} known sites | {len(tools)} tools "
      f"| {len(scores):,} score rows")

## 4. Evaluation grid


In [ ]:
def assign_third(positions, ref):
    start, end = CORE[ref]
    width = (end - start + 1) // 3
    offset = np.asarray(positions, dtype=int) - start
    if (offset < 0).any() or (np.asarray(positions, dtype=int) > end).any():
        raise ValueError(f"{ref}: position outside evaluated core")
    index = np.minimum(offset // width, 2)
    return pd.Categorical(np.asarray(THIRDS, dtype=object)[index], THIRDS, ordered=True)


def quartile_edges(values):
    edges = np.quantile(np.asarray(values, dtype=float), [0, .25, .5, .75, 1], method="linear")
    if not np.isfinite(edges).all() or not np.all(np.diff(edges) > 0):
        raise ValueError(f"Quartile cutoffs are not strictly increasing: {edges}")
    return edges


def apply_quartiles(values, edges):
    values = np.asarray(values, dtype=float)
    labels = np.select([values <= edges[1], values <= edges[2], values <= edges[3]],
                       QUARTILES[:3], default=QUARTILES[3])
    return pd.Categorical(labels, QUARTILES, ordered=True)


core_coverage = pd.concat([
    coverage.loc[(coverage["ref"] == ref) & coverage["pos"].between(start, end)].copy()
    for ref, (start, end) in CORE.items()
], ignore_index=True)
core_coverage["third"] = pd.concat([
    pd.Series(assign_third(block["pos"], ref), index=block.index)
    for ref, block in core_coverage.groupby("ref", sort=False)
]).sort_index()

cut_records = []
for (ref, rep), block in core_coverage.groupby(["ref", "rep"], sort=True):
    for descriptor in ["limiting", "abs_log2_ratio"]:
        edges = quartile_edges(block[descriptor])
        cut_records.append({"ref": ref, "rep": rep, "descriptor": descriptor,
                            "minimum": edges[0], "q25": edges[1], "q50": edges[2],
                            "q75": edges[3], "maximum": edges[4]})
cutpoints = pd.DataFrame(cut_records)

frames = []
for ref, (start, end) in CORE.items():
    positions = pd.DataFrame({"canonical_pos": np.arange(start, end + 1), "ref": ref})
    positions["third"] = assign_third(positions["canonical_pos"], ref)
    frames.append(positions.merge(pd.DataFrame({"rep": REPS}), how="cross")
                  .merge(tools[["tool", "display", "order"]], how="cross"))

grid = pd.concat(frames, ignore_index=True).merge(offsets, on=["tool", "ref"],
                                                  how="left", validate="many_to_one")
grid["scored_pos"] = grid["canonical_pos"] + grid["peak_delta"]
grid = grid.merge(coverage.rename(columns={"pos": "scored_pos"}),
                  on=["ref", "rep", "scored_pos"], how="left", validate="many_to_one")
if grid[["native", "ivt", "limiting", "abs_log2_ratio"]].isna().any().any():
    raise ValueError("Coverage is missing at one or more tool-scored coordinates.")

cut_lookup = cutpoints.set_index(["descriptor", "ref", "rep"])
out_of_range = {}
for descriptor, target in [("limiting", "depth_q"), ("abs_log2_ratio", "imbal_q")]:
    grid[target] = None
    outside = 0
    for (ref, rep), index in grid.groupby(["ref", "rep"], sort=True).groups.items():
        row = cut_lookup.loc[(descriptor, ref, rep)]
        edges = row[["minimum", "q25", "q50", "q75", "maximum"]].to_numpy(float)
        values = grid.loc[index, descriptor].to_numpy(float)
        outside += int(((values < edges[0]) | (values > edges[4])).sum())
        grid.loc[index, target] = apply_quartiles(grid.loc[index, descriptor], edges).astype(str)
    grid[target] = pd.Categorical(grid[target], QUARTILES, ordered=True)
    out_of_range[target] = outside
# Tool-scored coordinates can sit just outside the canonical core, i.e. outside the range
# the cutoffs were computed from; apply_quartiles bins such values into Q1/Q4.
print("grid values outside canonical cutoff range (binned to Q1/Q4):", out_of_range)

truth_pos = {r: np.array(sorted(x["canonical_pos"])) for r, x in truth.groupby("ref")}
truth_set = {r: set(v) for r, v in truth_pos.items()}
grid["is_truth"] = [p in truth_set[r] for r, p in zip(grid["ref"], grid["canonical_pos"])]
grid["distance_to_truth"] = [int(np.abs(truth_pos[r] - p).min())
                             for r, p in zip(grid["ref"], grid["canonical_pos"])]

score_fields = keys + ["scored_pos", "score_metric", "threshold", "called_positive",
                       "threshold_roundtrip_rescued"]
grid = grid.merge(scores[score_fields].rename(columns={"called_positive": "score_positive"}),
                  on=keys + ["scored_pos"], how="left", validate="one_to_one")
grid["has_row"] = grid["score_metric"].notna()
grid["score_positive"] = grid["score_positive"].map(
    lambda value: bool(value) if pd.notna(value) else False)
grid = grid.merge(calls[keys].assign(call_event=True), on=keys, how="left",
                  validate="one_to_one")
grid["called"] = grid["call_event"].map(
    lambda value: bool(value) if pd.notna(value) else False)
grid = grid.drop(columns="call_event")
if not grid["called"].equals(grid["score_positive"]):
    raise ValueError("Evaluation grid calls disagree with pinned score decisions.")
grid["is_fp"] = grid["called"] & ~grid["is_truth"]
grid["is_tp"] = grid["called"] & grid["is_truth"]

for descriptor in ["native", "ivt", "limiting", "abs_log2_ratio"]:
    grid[f"{descriptor}_pct_in_third"] = grid.groupby(
        ["tool", "ref", "rep", "third"], observed=True)[descriptor].rank(pct=True)

assert len(grid) == 133380
assert int((~grid["is_truth"]).sum()) == 132300
assert int(grid["called"].sum()) == 1538
assert int(grid["is_fp"].sum()) == 1033
assert int(grid["is_tp"].sum()) == 505
assert int(grid["has_row"].sum()) == len(scores)
print(f"{len(grid):,} tool opportunities | {int(grid['is_fp'].sum()):,} false positives "
      f"| {int(grid['is_tp'].sum())} truth-site call events")


## 5. Scopes and FP rate


In [ ]:
SCOPES = {
    SCOPE_EXACT: grid,
    SCOPE_PM5: grid.loc[grid["distance_to_truth"] > NEAR_TRUTH_NT],
}


def fp_rates(rows, by, scope):
    negatives = rows.loc[~rows["is_truth"]]
    out = negatives.groupby(by, observed=True).agg(
        fp_calls=("is_fp", "sum"),
        opportunities=("is_fp", "size"),
        scored_opportunities=("has_row", "sum"),
    ).reset_index()
    out["fp_per_kb"] = 1000 * out["fp_calls"] / out["opportunities"]
    out["fp_per_kb_scored_only"] = np.where(
        out["scored_opportunities"] > 0,
        1000 * out["fp_calls"] / out["scored_opportunities"], np.nan)
    out["scored_fraction"] = out["scored_opportunities"] / out["opportunities"]
    out.insert(0, "scope", scope)
    return out


print({scope: {"opportunities": int((~rows["is_truth"]).sum()),
               "false_positives": int(rows.loc[~rows["is_truth"], "is_fp"].sum())}
       for scope, rows in SCOPES.items()})

## 6. False positives by transcript third


In [ ]:
truth_tool = (grid.loc[grid["is_truth"]]
              .groupby(["tool", "display", "order", "ref", "canonical_pos", "third"],
                       observed=True)["called"].sum()
              .rename("replicates_called").reset_index())
truth_tool["recovered_2_of_3"] = truth_tool["replicates_called"] >= 2
assert len(truth_tool) == 360 and int(truth_tool["recovered_2_of_3"].sum()) == 167

n_tools = (truth_tool.groupby(["ref", "canonical_pos"], observed=True)["recovered_2_of_3"]
           .sum().rename("n_tools").reset_index())
site_detection = truth.merge(n_tools, on=["ref", "canonical_pos"], how="left",
                             validate="one_to_one")
site_detection["n_tools"] = site_detection["n_tools"].fillna(0).astype(int)
site_detection["third"] = [assign_third([p], r)[0]
                           for r, p in zip(site_detection["ref"], site_detection["canonical_pos"])]
site_detection["third"] = pd.Categorical(site_detection["third"], THIRDS, ordered=True)
sites_per_third = site_detection.groupby("third", observed=True).size().reindex(THIRDS)

by_third = pd.concat([fp_rates(rows, ["third"], scope) for scope, rows in SCOPES.items()],
                     ignore_index=True)
site_third_summary = (site_detection.groupby("third", observed=True)
                      .agg(known_sites=("n_tools", "size"),
                           sites_detected=("n_tools", lambda s: int((s > 0).sum())))
                      .reset_index())
by_third = by_third.merge(site_third_summary, on="third", how="left")
by_third["third"] = pd.Categorical(by_third["third"], THIRDS, ordered=True)
by_third.sort_values(["scope", "third"])[
    ["scope", "third", "fp_calls", "opportunities", "fp_per_kb",
     "known_sites", "sites_detected"]]


## 7. False positives by coverage


In [ ]:
def joint(rows, quartile_column, scope):
    out = (fp_rates(rows, ["third", quartile_column], scope)
           .rename(columns={quartile_column: "quartile"})
           .set_index(["third", "quartile"])
           .reindex(pd.MultiIndex.from_product([THIRDS, QUARTILES],
                                               names=["third", "quartile"]))
           .reset_index())
    out["scope"] = scope
    for column in ["fp_calls", "opportunities", "scored_opportunities"]:
        out[column] = out[column].fillna(0).astype(int)
    out["thin_cell"] = out["opportunities"].between(1, MIN_OPPORTUNITIES - 1)
    out["no_opportunities"] = out["opportunities"].eq(0)
    return out


joint_coverage = pd.concat([joint(rows, "depth_q", scope)
                            for scope, rows in SCOPES.items()], ignore_index=True)
joint_imbalance = pd.concat([joint(rows, "imbal_q", scope)
                             for scope, rows in SCOPES.items()], ignore_index=True)

coverage_views = []
for scope, rows in SCOPES.items():
    marginal = fp_rates(rows, ["depth_q"], scope).rename(columns={"depth_q": "quartile"})
    marginal.insert(1, "position_context", "all thirds")
    within = fp_rates(rows.loc[rows["third"] == "3-prime"], ["depth_q"], scope).rename(
        columns={"depth_q": "quartile"})
    within.insert(1, "position_context", "within the 3-prime third")
    coverage_views.extend([marginal, within])
by_coverage = pd.concat(coverage_views, ignore_index=True)
by_coverage["thin_cell"] = by_coverage["opportunities"].between(1, MIN_OPPORTUNITIES - 1)
by_coverage["no_opportunities"] = by_coverage["opportunities"].eq(0)

exact_joint = joint_coverage.loc[joint_coverage["scope"] == SCOPE_EXACT]
exact_joint.pivot(index="third", columns="quartile", values="fp_per_kb").reindex(
    THIRDS, columns=QUARTILES)


## 8. False positives by native–IVT imbalance


In [ ]:
imbalance_tables = []
for scope, rows in SCOPES.items():
    quart = fp_rates(rows, ["imbal_q"], scope).rename(columns={"imbal_q": "stratum"})
    quart.insert(1, "stratification", "absolute-imbalance quartile")
    medians = (rows.loc[~rows["is_truth"]]
               .groupby("imbal_q", observed=True)["abs_log2_ratio"].median())
    quart["median_abs_log2_ratio"] = quart["stratum"].map(medians)

    fold_rows = rows.copy()
    fold_rows["imbalance_class"] = np.where(
        fold_rows["abs_log2_ratio"] > 1, ">2-fold", "within 2-fold")
    fold = fp_rates(fold_rows, ["imbalance_class"], scope).rename(
        columns={"imbalance_class": "stratum"})
    fold.insert(1, "stratification", "absolute twofold threshold")
    fold["median_abs_log2_ratio"] = (fold_rows.loc[~fold_rows["is_truth"]]
        .groupby("imbalance_class")["abs_log2_ratio"].median().reindex(fold["stratum"]).to_numpy())
    imbalance_tables.extend([quart, fold])

by_imbalance = pd.concat(imbalance_tables, ignore_index=True)
by_imbalance[["scope", "stratification", "stratum", "fp_calls", "opportunities",
              "fp_per_kb", "median_abs_log2_ratio"]]

## 9. Known sites detected by no tool


In [ ]:
canonical = core_coverage.copy()
for descriptor in ["native", "ivt", "limiting", "abs_log2_ratio"]:
    canonical[f"{descriptor}_pct_in_third"] = canonical.groupby(
        ["ref", "rep", "third"], observed=True)[descriptor].rank(pct=True)

canonical_truth = canonical.merge(
    truth, left_on=["ref", "pos"], right_on=["ref", "canonical_pos"],
    validate="many_to_one")
site_coverage = (canonical_truth.groupby(
    ["ref", "reference_label", "canonical_pos", "mature_pos", "modification_type",
     "site_label", "third"], observed=True)
    .agg(native_mean=("native", "mean"), native_min=("native", "min"),
         ivt_mean=("ivt", "mean"), ivt_min=("ivt", "min"),
         limiting_mean=("limiting", "mean"), limiting_min=("limiting", "min"),
         ivt_percentile_mean=("ivt_pct_in_third", "mean"),
         limiting_percentile_mean=("limiting_pct_in_third", "mean"),
         abs_log2_imbalance_mean=("abs_log2_ratio", "mean"),
         abs_log2_imbalance_max=("abs_log2_ratio", "max"),
         low_ivt_replicates=("ivt_pct_in_third", lambda s: int((s < .10).sum())),
         low_limiting_replicates=("limiting_pct_in_third", lambda s: int((s < .10).sum())),
         high_imbalance_replicates=("abs_log2_ratio", lambda s: int((s > 1).sum())))
    .reset_index())
site_coverage["low_ivt_2_of_3"] = site_coverage["low_ivt_replicates"] >= 2
site_coverage["low_limiting_2_of_3"] = site_coverage["low_limiting_replicates"] >= 2
site_coverage["high_imbalance_2_of_3"] = site_coverage["high_imbalance_replicates"] >= 2
sites = site_coverage.merge(n_tools, on=["ref", "canonical_pos"], validate="one_to_one")
if len(sites) != len(truth):
    raise ValueError("Site coverage table does not cover every known site.")

truth_grid = (grid.loc[grid["is_truth"]]
              .merge(truth, on=["ref", "canonical_pos"], validate="many_to_one"))
truth_grid["row_present_non_positive"] = truth_grid["has_row"] & ~truth_grid["called"]
truth_grid["absent_score_row"] = ~truth_grid["has_row"]
truth_grid["low_ivt"] = truth_grid["ivt_pct_in_third"] < .10
truth_grid["low_limiting"] = truth_grid["limiting_pct_in_third"] < .10
truth_grid["high_imbalance"] = truth_grid["abs_log2_ratio"] > 1

missed_keys = sites.loc[sites["n_tools"] == 0, ["ref", "canonical_pos"]]
missed_rows = truth_grid.merge(missed_keys, on=["ref", "canonical_pos"], validate="many_to_one")
missed_by_tool = (missed_rows.groupby(
    ["ref", "reference_label", "canonical_pos", "mature_pos", "modification_type",
     "site_label", "third", "tool", "display", "order", "peak_delta"], observed=True)
    .agg(scored_pos=("scored_pos", "first"),
         native_mean=("native", "mean"), native_min=("native", "min"),
         ivt_mean=("ivt", "mean"), ivt_min=("ivt", "min"),
         limiting_mean=("limiting", "mean"), limiting_min=("limiting", "min"),
         ivt_percentile_mean=("ivt_pct_in_third", "mean"),
         ivt_percentile_median=("ivt_pct_in_third", "median"),
         limiting_percentile_mean=("limiting_pct_in_third", "mean"),
         limiting_percentile_median=("limiting_pct_in_third", "median"),
         abs_log2_imbalance_mean=("abs_log2_ratio", "mean"),
         abs_log2_imbalance_max=("abs_log2_ratio", "max"),
         low_ivt_replicates=("low_ivt", "sum"),
         low_limiting_replicates=("low_limiting", "sum"),
         high_imbalance_replicates=("high_imbalance", "sum"),
         score_rows_present=("has_row", "sum"),
         row_present_non_positive=("row_present_non_positive", "sum"),
         absent_score_rows=("absent_score_row", "sum"),
         positive_calls=("called", "sum"))
    .reset_index().sort_values(["ref", "canonical_pos", "order"]))
missed_by_tool["low_ivt_2_of_3"] = missed_by_tool["low_ivt_replicates"] >= 2
missed_by_tool["low_limiting_2_of_3"] = missed_by_tool["low_limiting_replicates"] >= 2
missed_by_tool["high_imbalance_2_of_3"] = missed_by_tool["high_imbalance_replicates"] >= 2
missed_by_tool["recovered_2_of_3"] = missed_by_tool["positive_calls"] >= 2

assert len(missed_by_tool) == 30
assert int(missed_by_tool["score_rows_present"].sum()) == 81
assert int(missed_by_tool["absent_score_rows"].sum()) == 9
assert int(missed_by_tool["positive_calls"].sum()) == 0

missed_site_tool_summary = (missed_by_tool.groupby(
    ["ref", "reference_label", "canonical_pos", "mature_pos", "modification_type"],
    observed=True)
    .agg(tools_low_ivt_2_of_3=("low_ivt_2_of_3", "sum"),
         tools_low_limiting_2_of_3=("low_limiting_2_of_3", "sum"),
         tools_high_imbalance_2_of_3=("high_imbalance_2_of_3", "sum"),
         score_rows_present=("score_rows_present", "sum"),
         absent_score_rows=("absent_score_rows", "sum"))
    .reset_index())
missed_depth_summary = (missed_by_tool.groupby(["ref", "canonical_pos"], observed=True)
                        .agg(minimum_tool_scored_ivt_depth=("ivt_min", "min"),
                             minimum_tool_scored_limiting_depth=("limiting_min", "min"))
                        .reset_index())
missed_site_summary = (missed_site_tool_summary
    .merge(missed_depth_summary, on=["ref", "canonical_pos"], validate="one_to_one")
    .merge(sites[["ref", "canonical_pos", "native_mean", "ivt_mean", "limiting_mean",
                  "ivt_percentile_mean", "limiting_percentile_mean",
                  "low_ivt_2_of_3", "low_limiting_2_of_3", "high_imbalance_2_of_3"]],
           on=["ref", "canonical_pos"], validate="one_to_one"))

missed = sites.loc[sites["n_tools"] == 0]
found = sites.loc[sites["n_tools"] > 0]
print(f"{len(missed)} of {len(sites)} known sites detected by no tool")
print(f"canonical low limiting coverage in ≥2/3 replicates: "
      f"{int(missed['low_limiting_2_of_3'].sum())}/{len(missed)} missed versus "
      f"{int(found['low_limiting_2_of_3'].sum())}/{len(found)} recovered")
missed_site_tool_summary


## 10. Per-tool performance


In [ ]:
by_tool = pd.concat([fp_rates(rows, ["display", "third"], scope)
                     for scope, rows in SCOPES.items()], ignore_index=True)
by_tool = by_tool.rename(columns={"display": "tool"})

tool_third_index = pd.MultiIndex.from_product(
    [tools["display"], THIRDS], names=["display", "third"])
by_tool_recovery = (truth_tool.groupby(["display", "third"], observed=True)
                    .agg(sites_recovered=("recovered_2_of_3", "sum"),
                         known_sites_in_third=("canonical_pos", "size"))
                    .reindex(tool_third_index, fill_value=0).reset_index()
                    .rename(columns={"display": "tool"}))
by_tool_recovery["sensitivity"] = np.where(
    by_tool_recovery["known_sites_in_third"] > 0,
    by_tool_recovery["sites_recovered"] / by_tool_recovery["known_sites_in_third"], np.nan)

# One stable recovery stratum per tool × truth site: average the three replicate depths,
# then compare with that tool/reference's replicate-mean background distribution.
position_means = (grid.groupby(
    ["tool", "display", "order", "ref", "canonical_pos", "third"], observed=True)
    .agg(limiting_scored_mean=("limiting", "mean")).reset_index())
position_means["recovery_depth_q"] = None
for (tool, ref), index in position_means.groupby(["tool", "ref"], sort=True).groups.items():
    edges = quartile_edges(position_means.loc[index, "limiting_scored_mean"])
    position_means.loc[index, "recovery_depth_q"] = apply_quartiles(
        position_means.loc[index, "limiting_scored_mean"], edges).astype(str)
position_means["recovery_depth_q"] = pd.Categorical(
    position_means["recovery_depth_q"], QUARTILES, ordered=True)

truth_tool_depth = truth_tool.merge(
    position_means[["tool", "ref", "canonical_pos", "limiting_scored_mean",
                    "recovery_depth_q"]],
    on=["tool", "ref", "canonical_pos"], validate="one_to_one")
recovery_depth = (truth_tool_depth.groupby(["display", "recovery_depth_q"], observed=True)
                  .agg(truth_sites=("canonical_pos", "size"),
                       sites_recovered=("recovered_2_of_3", "sum")).reset_index()
                  .rename(columns={"display": "tool", "recovery_depth_q": "quartile"}))
coverage_index = pd.MultiIndex.from_product(
    [tools["display"], QUARTILES], names=["tool", "quartile"])
recovery_depth = (recovery_depth.set_index(["tool", "quartile"])
                  .reindex(coverage_index, fill_value=0).reset_index())
recovery_depth["sensitivity"] = np.where(
    recovery_depth["truth_sites"] > 0,
    recovery_depth["sites_recovered"] / recovery_depth["truth_sites"], np.nan)

fp_depth = fp_rates(SCOPES[SCOPE_EXACT], ["display", "depth_q"], SCOPE_EXACT).rename(
    columns={"display": "tool", "depth_q": "quartile",
             "opportunities": "negative_opportunities",
             "scored_opportunities": "row_present_negative_opportunities",
             "fp_per_kb": "fp_per_kb_full_universe",
             "fp_per_kb_scored_only": "fp_per_kb_row_present",
             "scored_fraction": "callability"})
tool_coverage = fp_depth.merge(recovery_depth, on=["tool", "quartile"],
                               validate="one_to_one")
tool_coverage = (tool_coverage.merge(
    tools[["display", "order"]].rename(columns={"display": "tool"}),
    on="tool", validate="many_to_one")
    .sort_values(["order", "quartile"]).reset_index(drop=True))
tool_coverage.insert(2, "order", tool_coverage.pop("order"))
tool_coverage.insert(3, "coverage_metric", "min(native, IVT)")
tool_coverage.insert(4, "fp_coordinate_basis", "tool-scored; per-replicate canonical cutoffs")
tool_coverage.insert(5, "recovery_coordinate_basis",
                     "tool-scored replicate mean; tool-reference cutoffs")

assert len(tool_coverage) == 40
assert int(tool_coverage["fp_calls"].sum()) == 1033
assert int(tool_coverage["truth_sites"].sum()) == 360
assert int(tool_coverage["sites_recovered"].sum()) == 167
qcheck = tool_coverage.groupby("quartile", observed=True)[["truth_sites", "sites_recovered"]].sum()
assert qcheck["truth_sites"].tolist() == [17, 63, 204, 76]
assert qcheck["sites_recovered"].tolist() == [7, 30, 97, 33]

mean_sensitivity = (by_tool_recovery.groupby("third", observed=True)["sensitivity"]
                    .mean().reindex(THIRDS))
print("mean tool sensitivity by third:", mean_sensitivity.round(3).to_dict())
qcheck.assign(sensitivity=qcheck["sites_recovered"] / qcheck["truth_sites"])

## 11. Figure


In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7.2, "axes.titlesize": 7.8, "axes.labelsize": 7.1,
    "xtick.labelsize": 6.8, "ytick.labelsize": 6.8, "legend.fontsize": 6.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 130, "savefig.dpi": 400, "savefig.bbox": "tight"})


def heat(ax, values, title, xlabels, ylabels, cmap=HEAT, vmax=None, counts=None,
         thin=None, annotations=None, fmt="{:.1f}", label_size=6.6):
    m = values.to_numpy(float)
    finite = m[np.isfinite(m)]
    vmax = vmax if vmax is not None else (np.nanmax(finite) if len(finite) else 1)
    ax.imshow(np.ma.masked_invalid(m), cmap=cmap, aspect="auto", vmin=0, vmax=vmax)
    ax.set_xticks(range(m.shape[1]), xlabels)
    ax.set_yticks(range(m.shape[0]), ylabels)
    count_array = counts.to_numpy() if counts is not None else None
    thin_array = thin.to_numpy(bool) if thin is not None else np.zeros_like(m, dtype=bool)
    annotation_array = annotations.to_numpy(object) if annotations is not None else None
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            if np.isnan(m[i, j]):
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor="#EDEFF1",
                                           edgecolor="white", lw=1.2))
                ax.text(j, i, "no\nopportunities", ha="center", va="center",
                        fontsize=5.5, color="#7C868C")
                continue
            dark = m[i, j] > .55 * vmax
            label = annotation_array[i, j] if annotation_array is not None else fmt.format(m[i, j])
            ax.text(j, i - (.10 if count_array is not None else 0), label,
                    ha="center", va="center", fontsize=label_size,
                    fontweight="bold" if count_array is not None else "normal",
                    color="white" if dark else INK)
            if count_array is not None:
                ax.text(j, i + .30, f"n={int(count_array[i, j]):,}", ha="center",
                        va="center", fontsize=5.5, color="white" if dark else "#5E6870")
            if thin_array[i, j]:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor="none",
                                           edgecolor="#999", lw=.2, hatch="//"))
    ax.set_title(title, loc="left", fontweight="bold", pad=8)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

def as_matrix(frame, values="fp_per_kb", columns="quartile", fill=None):
    out = frame.pivot(index="third", columns=columns, values=values).reindex(
        THIRDS, columns=QUARTILES)
    return out if fill is None else out.fillna(fill)

In [ ]:
depth_exact = joint_coverage.loc[joint_coverage["scope"] == SCOPE_EXACT]
imbal_exact = joint_imbalance.loc[joint_imbalance["scope"] == SCOPE_EXACT]
depth_rate = as_matrix(depth_exact)
depth_opp = as_matrix(depth_exact, "opportunities", fill=0)
depth_thin = as_matrix(depth_exact, "thin_cell", fill=False)
imbal_rate = as_matrix(imbal_exact)
imbal_opp = as_matrix(imbal_exact, "opportunities", fill=0)
imbal_thin = as_matrix(imbal_exact, "thin_cell", fill=False)

tool_all = by_tool.loc[by_tool["scope"] == SCOPE_EXACT]
tool_fp = tool_all.pivot(index="tool", columns="third", values="fp_per_kb").reindex(
    tools["display"], columns=THIRDS)
callability = (tool_all.groupby("tool", sort=False)
               .apply(lambda d: d["scored_opportunities"].sum() / d["opportunities"].sum(),
                      include_groups=False).reindex(tools["display"]))
tool_sensitivity = (by_tool_recovery.pivot(index="tool", columns="third", values="sensitivity")
                    .reindex(tools["display"], columns=THIRDS) * 100)
tool_recovered = (by_tool_recovery.pivot(index="tool", columns="third", values="sites_recovered")
                  .reindex(tools["display"], columns=THIRDS))
tool_denominator = (by_tool_recovery.pivot(index="tool", columns="third",
                                           values="known_sites_in_third")
                    .reindex(tools["display"], columns=THIRDS))
recovery_annotations = tool_sensitivity.copy().astype(object)
for i in range(len(recovery_annotations.index)):
    for j in range(len(recovery_annotations.columns)):
        recovery_annotations.iat[i, j] = (
            f"{int(tool_recovered.iat[i, j])}/{int(tool_denominator.iat[i, j])} "
            f"({tool_sensitivity.iat[i, j]:.0f}%)")

supported_rates = pd.concat([
    depth_exact.loc[~depth_exact["thin_cell"], "fp_per_kb"],
    imbal_exact.loc[~imbal_exact["thin_cell"], "fp_per_kb"]]).dropna()
vmax = max(1.0, supported_rates.max())
band = [TLAB[t] for t in THIRDS]
order = tools["display"].tolist()

third_exact = by_third.loc[by_third["scope"] == SCOPE_EXACT].set_index("third")
third_pm5 = by_third.loc[by_third["scope"] == SCOPE_PM5].set_index("third")
near_share = 100 * grid.loc[grid["is_fp"], "distance_to_truth"].le(NEAR_TRUTH_NT).mean()

fig = plt.figure(figsize=(7.1, 9.7))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=.82, wspace=.42,
                       height_ratios=[1, 1.22, 1.0, 1.0])

ax_q1 = fig.add_subplot(gs[0, 0])
heat(ax_q1, depth_rate, "a   FP per 1,000 positions, by depth",
     ["Q1\nlowest", "Q2", "Q3", "Q4\nhighest"], band,
     vmax=vmax, counts=depth_opp, thin=depth_thin)
heat(fig.add_subplot(gs[0, 1]), imbal_rate,
     "b   FP per 1,000 positions, by depth imbalance",
     ["Q1\nleast", "Q2", "Q3", "Q4\nmost"], band,
     vmax=vmax, counts=imbal_opp, thin=imbal_thin)

ax_q2 = fig.add_subplot(gs[1, 0])
heat(ax_q2, tool_fp, "c   FP per 1,000 positions, by tool", band,
     [f"{n}  ({100 * callability[n]:.0f}% scored)" for n in order], label_size=6.2)
heat(fig.add_subplot(gs[1, 1]), tool_sensitivity,
     "d   sites detected by tool (≥2/3 replicates)",
     [f"{TLAB[t]}\n({int(sites_per_third[t])} sites)" for t in THIRDS], order,
     cmap="Greens", vmax=100, annotations=recovery_annotations, label_size=6.0)

ax_q3 = fig.add_subplot(gs[2, :])
found_s, missed_s = sites.loc[sites["n_tools"] > 0], sites.loc[sites["n_tools"] == 0]
ax_q3.scatter(found_s["limiting_mean"], found_s["abs_log2_imbalance_mean"], s=46,
              c="#B9C6CE", edgecolor="#8A99A3", linewidth=.7, zorder=2,
              label=f"recovered by ≥1 tool  (n={len(found_s)})")
ax_q3.scatter(missed_s["limiting_mean"], missed_s["abs_log2_imbalance_mean"], s=76,
              facecolor=IVT_C, edgecolor="white", linewidth=1.4, zorder=3,
              label=f"recovered by no tool  (n={len(missed_s)})")
missed_label_offsets = {
    ("16s", 1407): (0, -12, "left"),
    ("23s", 1618): (-7, 9, "right"),
    ("23s", 1939): (-20, 4, "center"),
}
for row in missed_s.itertuples(index=False):
    xoff, yoff, align = missed_label_offsets[(row.ref, int(row.mature_pos))]
    ax_q3.annotate(f"{row.reference_label.split()[0]} {row.modification_type}{int(row.mature_pos)}",
                   (row.limiting_mean, row.abs_log2_imbalance_mean),
                   textcoords="offset points", xytext=(xoff, yoff), ha=align, fontsize=6.2,
                   color=IVT_C, fontweight="bold")
ax_q3.set_xlabel("Mean lower depth at the canonical site")
ax_q3.set_ylabel("native-IVT coverage imbalance")
ax_q3.set_title("e   Coverage at the 36 known sites", loc="left", fontweight="bold")
ax_q3.set_xlim(0, sites["limiting_mean"].max() * 1.1)
ax_q3.legend(frameon=False, loc="upper right")

ax_ctx = fig.add_subplot(gs[3, :])
for ref, colour in zip(CORE, [NATIVE_C, IVT_C]):
    track = (coverage.loc[coverage["ref"] == ref].groupby("pos")["log2_ratio"]
             .mean().loc[CORE[ref][0]:CORE[ref][1]])
    ax_ctx.plot(np.linspace(0, 100, len(track)),
                track.rolling(51, center=True, min_periods=1).mean(),
                color=colour, lw=1.7, label=f"{ref.upper()} rRNA")
ax_ctx.axhline(0, color="#54606A", lw=1, ls="--")
ax_ctx.annotate("equal depth", (99, 0), textcoords="offset points",
                xytext=(0, 5), ha="right", fontsize=7.5, color="#6C767C")
ax_ctx.set_xlabel("Position along the transcript  (%, 5′ → 3′)")
ax_ctx.set_ylabel("log2((native+1)/(IVT+1))\n(above 0 = native deeper)")
ax_ctx.set_title("f   Native–IVT depth direction along each transcript",
                 loc="left", fontweight="bold")
ax_ctx.legend(frameon=False, loc="upper right")

for suffix in ("png", "pdf", "svg"):
    fig.savefig(OUTDIR / f"SuppFigS11_ivt_coverage_control.{suffix}")
plt.show()

## 12. Save tables


In [ ]:
by_third_rep = pd.concat([fp_rates(rows, ["third", "rep"], scope)
                          for scope, rows in SCOPES.items()], ignore_index=True)

SCOPE_MAP = {
    "exact-match benchmark universe": "Exact-match benchmark",
    SCOPE_PM5: "Excluding ±5 nt of a known site",
}
REGION_MAP = {"5-prime": "5'", "middle": "Middle", "3-prime": "3'"}
CONTEXT_MAP = {
    "all thirds": "All regions",
    "within the 3-prime third": "Within 3' region",
}
STRAT_MAP = {
    "absolute-imbalance quartile": "Quartile",
    "absolute twofold threshold": "Twofold threshold",
}


def write_csv(frame, name):
    frame.to_csv(OUTDIR / f"{name}.csv", index=False)


rep_wide = (
    by_third_rep.pivot_table(
        index=["scope", "third"], columns="rep", values="fp_per_kb", aggfunc="first"
    )
    .rename(columns={"rep1": "fp_rep1", "rep2": "fp_rep2", "rep3": "fp_rep3"})
    .reset_index()
)
s11 = by_third.merge(rep_wide, on=["scope", "third"], how="left")
s11_out = pd.DataFrame({
    "Scope": s11["scope"].map(SCOPE_MAP),
    "Region": s11["third"].map(REGION_MAP),
    "False-Positive Calls": s11["fp_calls"],
    "Tested Positions": s11["opportunities"],
    "Scored Tested Positions": s11["scored_opportunities"],
    "False Positives per 1,000 Tested Positions": s11["fp_per_kb"],
    "False Positives per 1,000 Scored Positions": s11["fp_per_kb_scored_only"],
    "Scored Fraction": s11["scored_fraction"],
    "Known Sites": s11["known_sites"],
    "Sites Detected": s11["sites_detected"],
    "False Positives per 1,000 Tested Positions (Replicate 1)": s11["fp_rep1"],
    "False Positives per 1,000 Tested Positions (Replicate 2)": s11["fp_rep2"],
    "False Positives per 1,000 Tested Positions (Replicate 3)": s11["fp_rep3"],
})

depth = by_coverage.copy()
imbal = by_imbalance.copy()
s12_depth = pd.DataFrame({
    "Scope": depth["scope"].map(SCOPE_MAP),
    "Analysis": "Limiting depth",
    "Subset": depth["position_context"].map(CONTEXT_MAP),
    "Stratum": depth["quartile"],
    "False-Positive Calls": depth["fp_calls"],
    "Tested Positions": depth["opportunities"],
    "Scored Tested Positions": depth["scored_opportunities"],
    "False Positives per 1,000 Tested Positions": depth["fp_per_kb"],
    "False Positives per 1,000 Scored Positions": depth["fp_per_kb_scored_only"],
    "Scored Fraction": depth["scored_fraction"],
    "Sparse Cell": depth["thin_cell"].map({True: "Yes", False: "No"}),
    "Median Absolute log2 Native-IVT Ratio": np.nan,
})
s12_imbal = pd.DataFrame({
    "Scope": imbal["scope"].map(SCOPE_MAP),
    "Analysis": "Native-IVT imbalance",
    "Subset": imbal["stratification"].map(STRAT_MAP),
    "Stratum": imbal["stratum"],
    "False-Positive Calls": imbal["fp_calls"],
    "Tested Positions": imbal["opportunities"],
    "Scored Tested Positions": imbal["scored_opportunities"],
    "False Positives per 1,000 Tested Positions": imbal["fp_per_kb"],
    "False Positives per 1,000 Scored Positions": imbal["fp_per_kb_scored_only"],
    "Scored Fraction": imbal["scored_fraction"],
    "Sparse Cell": np.nan,
    "Median Absolute log2 Native-IVT Ratio": imbal["median_abs_log2_ratio"],
})
s12_out = pd.concat([s12_depth, s12_imbal], ignore_index=True)

pooled = (
    tool_coverage.groupby("quartile", as_index=False)
    .agg(truth_sites=("truth_sites", "sum"), sites_recovered=("sites_recovered", "sum"))
)
pooled["quartile"] = pd.Categorical(pooled["quartile"], QUARTILES, ordered=True)
pooled = pooled.sort_values("quartile").reset_index(drop=True)
pooled["sensitivity"] = pooled["sites_recovered"] / pooled["truth_sites"]
s13_out = pd.DataFrame({
    "Depth Quartile": pooled["quartile"].astype(str),
    "Tool-Site Pairs": pooled["truth_sites"],
    "Sites Recovered": pooled["sites_recovered"],
    "Sensitivity": pooled["sensitivity"],
})

s14 = missed_site_summary
s14_out = pd.DataFrame({
    "rRNA": s14["reference_label"],
    "Mature Position": s14["mature_pos"],
    "Modification": s14["modification_type"],
    "Site": [
        f"{lab.split()[0]} {mod}{int(pos)}"
        for lab, mod, pos in zip(s14["reference_label"], s14["modification_type"], s14["mature_pos"])
    ],
    "Tools with Low Relative IVT Coverage (of 10)": s14["tools_low_ivt_2_of_3"],
    "Tools with Low Limiting Depth (of 10)": s14["tools_low_limiting_2_of_3"],
    "Tools with High Imbalance (of 10)": s14["tools_high_imbalance_2_of_3"],
    "Score Rows Present (of 30)": s14["score_rows_present"],
    "Absent Score Rows": s14["absent_score_rows"],
    "Minimum Tool-Scored IVT Depth": s14["minimum_tool_scored_ivt_depth"],
    "Minimum Tool-Scored Limiting Depth": s14["minimum_tool_scored_limiting_depth"],
    "Mean Native Depth": s14["native_mean"],
    "Mean IVT Depth": s14["ivt_mean"],
    "Mean Limiting Depth": s14["limiting_mean"],
})

for frame, name in [
    (s11_out, "Supplementary_Table_S11_False_Positives_by_Region"),
    (s12_out, "Supplementary_Table_S12_Coverage_and_Imbalance_Strata"),
    (s13_out, "Supplementary_Table_S13_Recovery_by_Depth_Quartile"),
    (s14_out, "Supplementary_Table_S14_Persistently_Undetected_Sites"),
]:
    write_csv(frame, name)

# Figure source tables (internal names unchanged)
for frame, name in [
    (joint_coverage, "fp_joint_third_by_limiting_depth"),
    (joint_imbalance, "fp_joint_third_by_native_ivt_ratio"),
    (by_tool, "fp_by_tool_and_third"),
    (by_tool_recovery, "recovery_by_tool_and_third"),
    (sites, "known_site_coverage_and_detection"),
]:
    write_csv(frame, name)

print(f"Wrote {len(list(OUTDIR.glob('*.csv')))} CSV tables to {OUTDIR}")
